# Causal Tracing (Standard)

Graph-first notebook for the standard workflow in `src/causal_trace/causal_trace.py`.

This is the non-alt variant. It corrupts every subject-token embedding, then restores each subject token at every layer. The main outputs are heatmaps and layer curves for visually inspecting where restoration recovers the clean target-token probability.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'src' / 'main.py').exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')


In [ ]:
import logging
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from hydra import compose, initialize_config_dir

from src.handlers.rome import ModelHandler
from src.common.loading import load_dataset, logits_to_probs, sample
from src.causal_trace.causal_trace import filter_dataset, preprocess_prompt

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.grid'] = True
print('Imports OK')


## 2. Load Config, Model, And Dataset

Set `MODEL_CONFIG` to a file from `src/config/model/` without `.yaml`. Keep `NUM_PROMPTS` small while inspecting; each prompt runs `subject_tokens x layers` restoration passes.


In [ ]:
MODEL_CONFIG = 'gpt2-large'
NUM_PROMPTS = 3

config_dir = str(ROOT / 'src' / 'config')
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(
        config_name='latium',
        overrides=[
            'command=causal_trace',
            f'model={MODEL_CONFIG}',
            f'generation.num_of_runs={NUM_PROMPTS}',
        ],
    )

print(f'Model: {cfg.model.name}')
print(f'Command: {cfg.command.name}')
print(f'Noise multiplier: {cfg.model.get("corruption_noise_multiplier", "auto")}')


In [ ]:
handler = ModelHandler(cfg)
print(f'Loaded {cfg.model.name} with {handler.num_of_layers} layers')

dataset = load_dataset(cfg)
df_dataset = filter_dataset(dataset['requested_rewrite'])
print(f'Dataset rows: {len(df_dataset)}')


## 3. Standard Trace Helper

Same restoration logic as `causal_trace_single_run`, but it returns matrices in memory for plotting instead of appending CSV rows.


In [ ]:
@dataclass
class StandardTraceResult:
    prompt_idx: int
    subject: str
    target: str
    clean_token: str
    corrupt_token: str
    clean_prob: float
    corrupt_prob: float
    subject_positions: list[int]
    restoration: dict[int, np.ndarray]

    @property
    def matrix(self) -> np.ndarray:
        return np.stack([self.restoration[pos] for pos in self.subject_positions], axis=0)

    @property
    def layer_mean(self) -> np.ndarray:
        return self.matrix.mean(axis=0)


def _decode_token(handler: ModelHandler, token_id: Any) -> str:
    return handler.tokenizer.batch_decode(token_id, skip_special_tokens=True)[0].strip()


def trace_prompt_standard(
    handler: ModelHandler,
    prompt_idx: int,
    subject: str,
    input_ids: Any,
    subject_positions: list[int],
    target: str,
) -> StandardTraceResult | None:
    with torch.no_grad():
        outputs_clean = handler.model(**input_ids, output_hidden_states=True, use_cache=False)
        next_token_id_clean = sample(outputs_clean['logits'][:, -1, :])
        clean_token = _decode_token(handler, next_token_id_clean)
        clean_prob = float(logits_to_probs(outputs_clean['logits'], next_token_id_clean).item())

        if clean_token != target:
            return None

        handler.set_corrupt_idx(subject_positions)
        handler.set_corrupt_hook()
        try:
            outputs_corrupt = handler.model(**input_ids, use_cache=False)
            next_token_id_corrupt = sample(outputs_corrupt['logits'][:, -1, :])
            corrupt_token = _decode_token(handler, next_token_id_corrupt)
            corrupt_prob = float(logits_to_probs(outputs_corrupt['logits'], next_token_id_clean).item())
        finally:
            handler.remove_hooks()

        restoration: dict[int, np.ndarray] = {}
        for restore_token_idx in subject_positions:
            probs = []
            handler.set_corrupt_idx(subject_positions)
            handler.set_corrupt_hook()
            try:
                for restore_layer in range(handler.num_of_layers):
                    handler.set_restore_idx(restore_token_idx)
                    handler.set_restore_layer(restore_layer)
                    restore_point = outputs_clean['hidden_states'][restore_layer + 1][0][restore_token_idx, :]
                    handler.set_restore_point(restore_point)
                    handler.set_restore_hook()
                    try:
                        outputs_restore = handler.model(**input_ids, use_cache=False)
                        prob = logits_to_probs(outputs_restore['logits'], next_token_id_clean).item()
                        probs.append(float(prob))
                    finally:
                        handler.unset_restore_hook()
            finally:
                handler.remove_hooks()
            restoration[int(restore_token_idx)] = np.asarray(probs, dtype=float)

    return StandardTraceResult(
        prompt_idx=int(prompt_idx),
        subject=str(subject),
        target=str(target),
        clean_token=clean_token,
        corrupt_token=corrupt_token,
        clean_prob=clean_prob,
        corrupt_prob=corrupt_prob,
        subject_positions=[int(pos) for pos in subject_positions],
        restoration=restoration,
    )


## 4. Run Traces

In [ ]:
results: list[StandardTraceResult] = []
total = 0
failed = 0

for prompt_dict in df_dataset.itertuples():
    if len(results) >= NUM_PROMPTS:
        break
    total += 1

    preprocessed = preprocess_prompt(handler, prompt_dict)
    if preprocessed is None:
        failed += 1
        continue

    prompt_ids, subject_positions = preprocessed
    result = trace_prompt_standard(
        handler,
        prompt_idx=prompt_dict.Index,
        subject=prompt_dict.subject,
        input_ids=prompt_ids,
        subject_positions=subject_positions,
        target=prompt_dict.target_true['str'],
    )

    if result is None:
        failed += 1
        print(f'SKIP clean-token mismatch: {prompt_dict.subject!r} -> {prompt_dict.target_true["str"]!r}')
        continue

    results.append(result)
    peak = int(np.argmax(result.layer_mean))
    print(
        f'OK {len(results):02d}: {result.subject!r} -> {result.target!r}  '
        f'clean={result.clean_prob:.4f} corrupt={result.corrupt_prob:.4f} peak=L{peak}'
    )

print(f'Done: {len(results)} successful, {failed} failed, {total} attempted')


## 5. Prompt Gallery

Each prompt gets a token-position heatmap and a matching mean layer curve. The dashed line marks the peak layer for that prompt.


In [ ]:
if not results:
    raise RuntimeError('No successful traces. Try increasing NUM_PROMPTS or changing MODEL_CONFIG.')

fig, axes = plt.subplots(len(results), 2, figsize=(13, 3.8 * len(results)), squeeze=False)

for row, result in enumerate(results):
    matrix = result.matrix
    mean_curve = result.layer_mean
    peak_layer = int(np.argmax(mean_curve))

    heat_ax = axes[row, 0]
    image = heat_ax.imshow(matrix, aspect='auto', cmap='viridis')
    heat_ax.axvline(peak_layer, color='white', linestyle='--', linewidth=1.2)
    heat_ax.set_title(f'{result.subject} -> {result.target}: restored token x layer')
    heat_ax.set_xlabel('Layer')
    heat_ax.set_ylabel('Subject token position')
    heat_ax.set_yticks(range(len(result.subject_positions)))
    heat_ax.set_yticklabels(result.subject_positions)
    fig.colorbar(image, ax=heat_ax, fraction=0.046, pad=0.04)

    curve_ax = axes[row, 1]
    layers = np.arange(handler.num_of_layers)
    curve_ax.plot(layers, mean_curve, marker='o', color='steelblue')
    curve_ax.axhline(result.clean_prob, color='seagreen', linestyle=':', label='clean prob')
    curve_ax.axhline(result.corrupt_prob, color='gray', linestyle=':', label='corrupt prob')
    curve_ax.axvline(peak_layer, color='crimson', linestyle='--', label=f'peak L{peak_layer}')
    curve_ax.set_title('Mean restoration over subject tokens')
    curve_ax.set_xlabel('Layer')
    curve_ax.set_ylabel('Target-token probability')
    curve_ax.set_xlim(-0.5, handler.num_of_layers - 0.5)
    curve_ax.legend(fontsize=8)

fig.suptitle(f'Standard causal tracing: {cfg.model.name}', fontsize=13)
plt.tight_layout()
plt.show()


## 6. Run-Level Graphs

In [ ]:
layer_curves = np.stack([result.layer_mean for result in results], axis=0)
avg_curve = layer_curves.mean(axis=0)
std_curve = layer_curves.std(axis=0)
layers = np.arange(handler.num_of_layers)
peak = int(np.argmax(avg_curve))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

heat = axes[0].imshow(layer_curves, aspect='auto', cmap='magma')
axes[0].set_title('Prompt x layer restoration heatmap')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Prompt')
axes[0].set_yticks(range(len(results)))
axes[0].set_yticklabels([f'{r.subject} -> {r.target}' for r in results], fontsize=8)
for idx, curve in enumerate(layer_curves):
    axes[0].plot(int(np.argmax(curve)), idx, marker='x', color='white', markersize=7)
fig.colorbar(heat, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].plot(layers, avg_curve, marker='o', color='steelblue', label='mean')
axes[1].fill_between(layers, avg_curve - std_curve, avg_curve + std_curve, color='steelblue', alpha=0.18, label='1 std')
axes[1].axvline(peak, color='crimson', linestyle='--', label=f'peak L{peak}')
axes[1].set_title(f'Average restoration curve over {len(results)} prompts')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Target-token probability')
axes[1].set_xlim(-0.5, handler.num_of_layers - 0.5)
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Summary And Optional Save

In [ ]:
summary = pd.DataFrame(
    {
        'prompt_idx': result.prompt_idx,
        'subject': result.subject,
        'target': result.target,
        'clean_prob': result.clean_prob,
        'corrupt_prob': result.corrupt_prob,
        'peak_layer': int(np.argmax(result.layer_mean)),
        'peak_prob': float(np.max(result.layer_mean)),
        'subject_positions': result.subject_positions,
    }
    for result in results
)
summary


In [ ]:
SAVE = False
OUTPUT_DIR = ROOT / 'analysis_out' / 'causal_trace_notebook'

if SAVE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model_slug = cfg.model.name.replace('/', '-')
    summary_path = OUTPUT_DIR / f'{model_slug}_standard_summary.csv'
    curve_path = OUTPUT_DIR / f'{model_slug}_standard_avg_curve.csv'
    summary.to_csv(summary_path, index=False)
    pd.DataFrame({'layer': layers, 'mean': avg_curve, 'std': std_curve}).to_csv(curve_path, index=False)
    print(f'Wrote {summary_path}')
    print(f'Wrote {curve_path}')
else:
    print('Set SAVE = True to write summary CSV files under analysis_out/causal_trace_notebook/')
